# Module 8 — Testing (pytest) + Logging/Monitoring
Exam domain: **Security, Governance, Monitoring & Testing**

Databricks notebook version. On Databricks, `pipeline_transforms.py` typically
lives in a Databricks Repo (Git-backed) and is imported like any Python module;
tests run via `%sh pytest` against the Repo checkout, or as a separate CI step
(see Module 9).

In [ ]:
%sh mkdir -p /tmp/module8 && cat > /tmp/module8/pipeline_transforms.py << 'EOF'
from pyspark.sql import functions as F

def clean_orders(orders_df, min_order_date="2024-01-01"):
    return (orders_df
        .filter(F.col("amount") > 0)
        .filter(F.col("order_date") >= F.lit(min_order_date)))

def enrich_with_customer(orders_df, customers_df):
    return orders_df.join(customers_df, on="customer_id", how="left")
EOF

In [ ]:
%sh cat > /tmp/module8/test_pipeline_transforms.py << 'EOF'
import pytest
from pyspark.sql import SparkSession
from pipeline_transforms import clean_orders, enrich_with_customer

@pytest.fixture(scope="module")
def spark():
    return SparkSession.builder.appName("pytest").master("local[2]").getOrCreate()

def test_clean_orders_drops_negative_amount(spark):
    df = spark.createDataFrame([(1, "2024-02-01", -5.0)], ["order_id", "order_date", "amount"])
    assert clean_orders(df).count() == 0
EOF

In [ ]:
%sh cd /tmp/module8 && python -m pytest test_pipeline_transforms.py -v

## Logging on Databricks
Same Python `logging` module works; on Databricks, cluster logs are also
delivered to DBFS/cloud storage if log delivery is configured, and structured
log lines can be shipped to a monitoring table for dashboards.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("module8_pipeline")
logger.info("Pipeline started on cluster")

## Native monitoring: Jobs UI + system tables
- **Workflows -> Jobs -> Runs**: duration, status, retries per task, per run —
  no extra code needed.
- **System tables** (`system.lakeflow.job_run_timeline`,
  `system.access.audit`, etc.) let you query job health and access history with
  SQL instead of parsing logs.
- **Alerts**: attach a SQL alert or a Job's built-in failure notification
  (email/Slack/webhook) instead of hand-rolling alerting logic.

In [ ]:
%sql
-- SELECT * FROM system.lakeflow.job_run_timeline WHERE result_state = 'FAILED' ORDER BY period_start_time DESC LIMIT 20;